In [3]:
print("OK")

OK


In [4]:
%pwd

'e:\\Projects\\Medical-Chatbot-with-LLMs-LangChain-and-PineCone\\research'

In [5]:
import os

os.chdir(r"E:\Projects\Medical-Chatbot-with-LLMs-LangChain-and-PineCone")

In [6]:
%pwd

'E:\\Projects\\Medical-Chatbot-with-LLMs-LangChain-and-PineCone'

In [7]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

e:\Projects\Medical-Chatbot-with-LLMs-LangChain-and-PineCone\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Extract test from PDF Files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [9]:
extracted_data = load_pdf_files("data")

In [10]:
extracted_data

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2015-02-17T16:24:08+02:00', 'moddate': '2015-02-20T11:39:51+01:00', 'trapped': '/False', 'source': 'data\\medical_book.pdf', 'total_pages': 180, 'page': 0, 'page_label': '1'}, page_content='THINKING HEALTHY\nA manual for psychosocial management of perinatal depression\nWHO generic field-trial version 1.0, 2015\nSeries on Low-Intensity Psychological Interventions – 1'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2015-02-17T16:24:08+02:00', 'moddate': '2015-02-20T11:39:51+01:00', 'trapped': '/False', 'source': 'data\\medical_book.pdf', 'total_pages': 180, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2015-02-17T16:24:08+02:00', 'moddate': '2015-02-20T11:39:51+01:00', 'trapp

In [11]:
len(extracted_data)

180

In [13]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Return documents containing only the sources metadata
    and the original page content.
    """
    minimal_docs = []

    for doc in docs:
        minimal_doc = Document(
            page_content=doc.page_content,
            metadata={"sources": doc.metadata.get("sources")}
        )
        minimal_docs.append(minimal_doc)

    return minimal_docs

In [14]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [15]:
minimal_docs

[Document(metadata={'sources': None}, page_content='THINKING HEALTHY\nA manual for psychosocial management of perinatal depression\nWHO generic field-trial version 1.0, 2015\nSeries on Low-Intensity Psychological Interventions – 1'),
 Document(metadata={'sources': None}, page_content=''),
 Document(metadata={'sources': None}, page_content='Thinking Healthy\nA manual for psychosocial management\nof perinatal depression\nWHO generic field-trial version 1.0, 2015\nSeries on Low-Intensity Psychological Interventions – 1'),
 Document(metadata={'sources': None}, page_content='WHO/MSD/MER/15.1\n© World Health Organization 2015\nAll rights reserved. Publications of the World Health Organization are available on the WHO \nwebsite (www.who.int) or can be purchased from WHO Press, World Health Organization, \n20 Avenue Appia, 1211 Geneva 27, Switzerland (tel.: +41 22 791 3264; fax: +41 22 791 4857;  \ne-mail: bookorders@who.int). \nRequests for permission to reproduce or translate WHO publication

## Chunking Part

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )

    texts_chunks = text_splitter.split_documents(minimal_docs)
    return texts_chunks

In [17]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")


Number of chunks: 324


In [18]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HUggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )

    return embeddings

embedding = download_embeddings()

C:\Users\Dureksha Arangala\AppData\Local\Temp\ipykernel_22448\1101833612.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [19]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [20]:
vector = embedding.embed_query("Hello World")
vector

[-0.03447720408439636,
 0.031023239716887474,
 0.00673496862873435,
 0.026108969002962112,
 -0.03936196118593216,
 -0.16030246019363403,
 0.06692393124103546,
 -0.0064414795488119125,
 -0.047450557351112366,
 0.014758911915123463,
 0.0708753690123558,
 0.05552756413817406,
 0.01919337548315525,
 -0.026251327246427536,
 -0.010109500028192997,
 -0.026940541341900826,
 0.022307470440864563,
 -0.02222665585577488,
 -0.14969269931316376,
 -0.01749308407306671,
 0.007676247972995043,
 0.054352279752492905,
 0.003254473675042391,
 0.03172597661614418,
 -0.0846213549375534,
 -0.0294059906154871,
 0.051595624536275864,
 0.048124030232429504,
 -0.003314792178571224,
 -0.05827920511364937,
 0.04196930304169655,
 0.022210685536265373,
 0.1281888484954834,
 -0.02233896590769291,
 -0.011656301096081734,
 0.06292833387851715,
 -0.032876282930374146,
 -0.09122605621814728,
 -0.031175389885902405,
 0.05269956216216087,
 0.0470348559319973,
 -0.08420302718877792,
 -0.03005620837211609,
 -0.0207447819411

In [21]:
print(f"Vector length: {len(vector)}")

Vector length: 384


In [22]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [23]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [24]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key = pinecone_api_key)

In [25]:
pc

In [26]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = len(vector),
        metric = "cosine",
        spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
    )

index = pc.Index(index_name)

In [27]:
for i, doc in enumerate(texts_chunk):
    print(i, doc.metadata)

0 {'sources': None}
1 {'sources': None}
2 {'sources': None}
3 {'sources': None}
4 {'sources': None}
5 {'sources': None}
6 {'sources': None}
7 {'sources': None}
8 {'sources': None}
9 {'sources': None}
10 {'sources': None}
11 {'sources': None}
12 {'sources': None}
13 {'sources': None}
14 {'sources': None}
15 {'sources': None}
16 {'sources': None}
17 {'sources': None}
18 {'sources': None}
19 {'sources': None}
20 {'sources': None}
21 {'sources': None}
22 {'sources': None}
23 {'sources': None}
24 {'sources': None}
25 {'sources': None}
26 {'sources': None}
27 {'sources': None}
28 {'sources': None}
29 {'sources': None}
30 {'sources': None}
31 {'sources': None}
32 {'sources': None}
33 {'sources': None}
34 {'sources': None}
35 {'sources': None}
36 {'sources': None}
37 {'sources': None}
38 {'sources': None}
39 {'sources': None}
40 {'sources': None}
41 {'sources': None}
42 {'sources': None}
43 {'sources': None}
44 {'sources': None}
45 {'sources': None}
46 {'sources': None}
47 {'sources': None}
48

In [28]:
for doc in texts_chunk:
    doc.metadata = {
        key: value
        for key, value in doc.metadata.items()
        if value is not None
    }

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [30]:
# Load Existing Index 

from langchain_pinecone import PineconeVectorStore

# Embed each chunk and upsert the embeddings into the Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding = embedding
)

# Add more data to the existing index

In [31]:
from langchain_core.documents import Document

dswith = Document(
    page_content="""
    More information about postpartum depression (PPD),including its symptoms, causes, risk factors, screening, treatment options, recovery support, and maternal wellbeing,
    can be found through reliable online medical resources.Information should be collected from trusted sources such as WHO, NHS, ACOG, and Postpartum Support International.
    """,
    metadata={
        "source": "Google",
        "category": "PPD Information"
    }
)

In [32]:
docsearch.add_documents(documents = [dswith])

['e0f35b1d-7f7b-4688-97fc-65049355e4b7']

In [33]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwards= {"k": 3}) # 3 relevant responses

In [34]:
retrived_docs = retriever.invoke("What is postpartum depression?")
retrived_docs

[Document(id='bb95981e-d3d7-40c2-b6b2-fa3ca7adfb10', metadata={'category': 'PPD Information', 'source': 'Google'}, page_content='\n    More information about postpartum depression (PPD),including its symptoms, causes, risk factors, screening, treatment options, recovery support, and maternal wellbeing,\n    can be found through reliable online medical resources.Information should be collected from trusted sources such as WHO, NHS, ACOG, and Postpartum Support International.\n    '),
 Document(id='cc4d8f55-828c-4c6d-87f7-d9369f7aed14', metadata={'category': 'PPD Information', 'source': 'Google'}, page_content='\n    More information about postpartum depression (PPD),including its symptoms, causes, risk factors, screening, treatment options, recovery support, and maternal wellbeing,\n    can be found through reliable online medical resources.Information should be collected from trusted sources such as WHO, NHS, ACOG, and Postpartum Support International.\n    '),
 Document(id='648ac370-1

# Connect to LLM

In [35]:
pip install langchain-ollama

In [36]:
from langchain_ollama import ChatOllama

chatmodel = ChatOllama(
model="llama3.2"
)

In [37]:
from langchain_ollama import ChatOllama

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

from langchain_pinecone import PineconeVectorStore

print("All imports successful!")

All imports successful!


In [38]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [39]:
question_answer_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [40]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [41]:
print(embedding)
print(type(embedding))

model='nomic-embed-text' base_url=None client_kwargs={} mirostat=None mirostat_eta=None mirostat_tau=None num_ctx=None num_gpu=None num_thread=None repeat_last_n=None repeat_penalty=None temperature=None stop=None tfs_z=None top_k=None top_p=None
<class 'langchain_ollama.embeddings.OllamaEmbeddings'>


In [43]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:latest",
    base_url="http://localhost:11434",
    temperature=0.3
)

In [45]:
response = rag_chain.invoke({"input": "What is Postpartum Depression?"})
print(response["answer"])

I don't know the answer to that question.


In [46]:
response = rag_chain.invoke({"input": "What are the treatment for PPD?"})
print(response["answer"])

I don't know the specific treatment options for postpartum depression (PPD) from the provided context. However, I can suggest that treatment options for PPD may include medication, therapy, and lifestyle changes, as recommended by trusted sources such as WHO, NHS, ACOG, and Postpartum Support International.
